# 🧠 Dense Retrieval with Embeddings - The Modern Approach

## What is Dense Retrieval?

**Dense retrieval uses neural networks to convert text into dense vectors (embeddings), then finds similar documents using vector similarity.**

### The Magic ✨

```python
Text: "The quick brown fox jumps"
    ↓ Embedding Model (BERT, E5, etc.)
Vector: [0.23, -0.45, 0.12, 0.89, ...] (768 dimensions)
```

**Similar meanings → Similar vectors!**

```
"dog" ≈ "puppy" ≈ "canine"  (close in vector space)
"dog" ≠ "computer"  (far in vector space)
```

## Why Dense Retrieval Wins 🏆

### Sparse (Keyword) vs Dense (Semantic)

| Query | Sparse Retrieval | Dense Retrieval |
|-------|-----------------|----------------|
| "NYC" | Misses "New York City" ❌ | Finds "New York City" ✅ |
| "automobile" | Misses "car" ❌ | Finds "car" ✅ |
| "programming" | Misses "coding" ❌ | Finds "coding" ✅ |

**Dense retrieval understands semantics, not just keywords!**

## Popular Embedding Models

### For RAG:
1. **Sentence-BERT** (all-MiniLM-L6-v2) - Fast, good quality
2. **E5** (intfloat/e5-base-v2) - State-of-the-art
3. **BGE** (BAAI/bge-base-en-v1.5) - High performance
4. **OpenAI** (text-embedding-ada-002) - Commercial
5. **Cohere** (embed-english-v3.0) - Commercial

### Model Specs:
```
all-MiniLM-L6-v2:  384 dims, 80M params,  14GB/s
E5-base-v2:        768 dims, 110M params, 10GB/s  
BGE-base:          768 dims, 110M params, 10GB/s
```

## 1. Your First Dense Retriever

In [1]:
# Install required packages
# !pip install sentence-transformers numpy

from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Model loaded: {model}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Max sequence length: {model.max_seq_length}")

Model loaded: SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
Embedding dimension: 384
Max sequence length: 256


In [2]:
# Encode some text
text = "Dense retrieval uses neural embeddings for semantic search."

embedding = model.encode(text)

print(f"Text: {text}")
print(f"\nEmbedding shape: {embedding.shape}")
print(f"First 10 values: {embedding[:10]}")
print(f"\nEmbedding stats:")
print(f"  Mean: {embedding.mean():.4f}")
print(f"  Std: {embedding.std():.4f}")
print(f"  Min: {embedding.min():.4f}")
print(f"  Max: {embedding.max():.4f}")

Text: Dense retrieval uses neural embeddings for semantic search.

Embedding shape: (384,)
First 10 values: [ 0.0066053  -0.08323619  0.03368907  0.01396596 -0.04005925  0.09240836
  0.03923687 -0.0092196   0.04261614 -0.07403307]

Embedding stats:
  Mean: 0.0006
  Std: 0.0510
  Min: -0.1464
  Max: 0.1469


## 2. Semantic Similarity

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

# Test semantic similarity
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",  # Same meaning, different words
    "Dogs are great pets",  # Different meaning
    "Machine learning is amazing",  # Very different
]

# Encode all sentences
embeddings = model.encode(sentences)

# Calculate similarity matrix
similarity_matrix = cosine_similarity(embeddings)

print("Semantic Similarity Matrix:\n")
print(f"{'':40} " + " ".join([f"S{i+1:}" for i in range(len(sentences))]))
print("="*60)

for i, sent in enumerate(sentences):
    scores = " ".join([f"{score:.2f}" for score in similarity_matrix[i]])
    print(f"{sent[:38]:<40} {scores}")

print("\n💡 Notice: S1 and S2 have high similarity despite different words!")

Semantic Similarity Matrix:

                                         S1 S2 S3 S4
The cat sat on the mat                   1.00 0.56 0.18 -0.03
A feline rested on the rug               0.56 1.00 0.29 -0.05
Dogs are great pets                      0.18 0.29 1.00 0.29
Machine learning is amazing              -0.03 -0.05 0.29 1.00

💡 Notice: S1 and S2 have high similarity despite different words!


## 3. Building a Dense Retrieval System

In [4]:
class DenseRetriever:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        """Initialize dense retriever with embedding model"""
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None
        
    def index(self, documents):
        """Create embeddings for all documents"""
        self.documents = documents
        print(f"Encoding {len(documents)} documents...")
        self.embeddings = self.model.encode(
            documents, 
            show_progress_bar=True,
            batch_size=32
        )
        print(f"✅ Indexed {len(documents)} documents")
        print(f"   Embedding shape: {self.embeddings.shape}")
        
    def search(self, query, top_k=5):
        """Search for most similar documents"""
        # Encode query
        query_embedding = self.model.encode(query)
        
        # Calculate cosine similarity
        scores = cosine_similarity(
            [query_embedding], 
            self.embeddings
        )[0]
        
        # Get top-k indices
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        # Return results
        results = []
        for rank, idx in enumerate(top_indices):
            results.append({
                'rank': rank + 1,
                'score': float(scores[idx]),
                'document': self.documents[idx],
                'index': int(idx)
            })
        
        return results

# Test the retriever
docs = [
    "Python is a popular programming language for data science and web development.",
    "Machine learning models can predict outcomes based on historical data.",
    "Natural language processing enables computers to understand human language.",
    "Deep neural networks have revolutionized computer vision and speech recognition.",
    "RAG systems combine retrieval and generation for better AI responses.",
    "Vector databases efficiently store and query high-dimensional embeddings.",
    "Transformers are the foundation of modern large language models.",
    "Fine-tuning adapts pre-trained models to specific downstream tasks.",
]

retriever = DenseRetriever()
retriever.index(docs)

Encoding 8 documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 8 documents
   Embedding shape: (8, 384)


In [5]:
# Test search
query = "How do AI models understand text?"
results = retriever.search(query, top_k=3)

print(f"Query: {query}\n")
print("Top Results:")
print("="*80)

for result in results:
    print(f"\nRank {result['rank']} (Score: {result['score']:.4f}):")
    print(f"  {result['document']}")

Query: How do AI models understand text?

Top Results:

Rank 1 (Score: 0.5272):
  Natural language processing enables computers to understand human language.

Rank 2 (Score: 0.4230):
  Transformers are the foundation of modern large language models.

Rank 3 (Score: 0.3379):
  RAG systems combine retrieval and generation for better AI responses.


## 4. Comparing Different Embedding Models

In [6]:
# Compare popular models
models_to_test = [
    'all-MiniLM-L6-v2',           # Fast and efficient
    'all-mpnet-base-v2',          # Better quality
    'multi-qa-MiniLM-L6-cos-v1',  # Optimized for Q&A
]

query = "What is machine learning?"
doc_to_match = "Machine learning models can predict outcomes based on historical data."

print(f"Query: {query}")
print(f"Document to match: {doc_to_match}\n")
print(f"{'Model':<30} {'Dimensions':<12} {'Similarity':<12} {'Time'}")
print("="*70)

import time

for model_name in models_to_test:
    model = SentenceTransformer(model_name)
    
    # Encode
    start = time.time()
    q_emb = model.encode(query)
    d_emb = model.encode(doc_to_match)
    elapsed = time.time() - start
    
    # Similarity
    sim = cosine_similarity([q_emb], [d_emb])[0][0]
    
    print(f"{model_name:<30} {len(q_emb):<12} {sim:<12.4f} {elapsed*1000:.1f}ms")

print("\n💡 Higher dimensions ≠ always better! Consider speed vs quality trade-off.")

Query: What is machine learning?
Document to match: Machine learning models can predict outcomes based on historical data.

Model                          Dimensions   Similarity   Time
all-MiniLM-L6-v2               384          0.5390       1526.2ms


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

multi-qa-MiniLM-L6-cos-v1      384          0.4245       70.6ms

💡 Higher dimensions ≠ always better! Consider speed vs quality trade-off.


## 5. Batch Processing for Efficiency

In [7]:
# Efficient batch encoding
large_corpus = [
    f"This is document number {i} about various topics in AI and machine learning."
    for i in range(1000)
]

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding 1000 documents...\n")

# Method 1: One by one (slow)
start = time.time()
embeddings_slow = [model.encode(doc) for doc in large_corpus[:100]]  # Only 100 for demo
time_slow = time.time() - start

# Method 2: Batch (fast)
start = time.time()
embeddings_fast = model.encode(large_corpus[:100], batch_size=32)
time_fast = time.time() - start

print(f"One-by-one (100 docs): {time_slow:.2f}s")
print(f"Batch processing (100 docs): {time_fast:.2f}s")
print(f"\nSpeedup: {time_slow/time_fast:.1f}x faster with batching!")
print(f"\n💡 Always use batch_size parameter for large datasets!")

Encoding 1000 documents...

One-by-one (100 docs): 4.45s
Batch processing (100 docs): 0.51s

Speedup: 8.8x faster with batching!

💡 Always use batch_size parameter for large datasets!


## 6. Advanced: Similarity Metrics Comparison

In [8]:
from sklearn.metrics.pairwise import euclidean_distances

# Different similarity metrics
query = "machine learning algorithms"
documents = [
    "Algorithms for machine learning and AI",
    "The weather is nice today",
    "Neural networks and deep learning techniques",
]

# Encode
query_emb = model.encode(query)
doc_embs = model.encode(documents)

# Calculate different similarities
cosine_sims = cosine_similarity([query_emb], doc_embs)[0]
dot_products = np.dot(doc_embs, query_emb)
euclidean_dists = euclidean_distances([query_emb], doc_embs)[0]

print(f"Query: {query}\n")
print(f"{'Document':<50} {'Cosine':<12} {'Dot Product':<12} {'Euclidean'}")
print("="*90)

for i, doc in enumerate(documents):
    print(f"{doc:<50} {cosine_sims[i]:<12.4f} {dot_products[i]:<12.4f} {euclidean_dists[i]:.4f}")

print("\n💡 Cosine similarity is most common for RAG (range: -1 to 1)")
print("   Dot product is faster but requires normalized vectors")
print("   Euclidean distance measures geometric distance (lower = more similar)")

Query: machine learning algorithms

Document                                           Cosine       Dot Product  Euclidean
Algorithms for machine learning and AI             0.8606       0.8606       0.5280
The weather is nice today                          0.0669       0.0669       1.3661
Neural networks and deep learning techniques       0.5714       0.5714       0.9259

💡 Cosine similarity is most common for RAG (range: -1 to 1)
   Dot product is faster but requires normalized vectors
   Euclidean distance measures geometric distance (lower = more similar)


## 7. Multi-lingual Embeddings

In [9]:
# Multilingual model
multilingual_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Same concept in different languages
texts = [
    "Machine learning is transforming AI",  # English
    "L'apprentissage automatique transforme l'IA",  # French
    "El aprendizaje automático está transformando la IA",  # Spanish
    "機械学習はAIを変革しています",  # Japanese
    "The weather is nice today",  # Different topic (English)
]

# Encode
embeddings = multilingual_model.encode(texts)

# Similarity matrix
sim_matrix = cosine_similarity(embeddings)

print("Multilingual Semantic Similarity:\n")
print(f"{'Text':<50} {'E1':<8} {'F2':<8} {'S3':<8} {'J4':<8} {'E5'}")
print("="*90)

for i, text in enumerate(texts):
    scores = " ".join([f"{sim_matrix[i][j]:.2f}" for j in range(len(texts))])
    print(f"{text[:48]:<50} {scores}")

print("\n💡 E1, F2, S3, J4 have high similarity - same meaning, different languages!")
print("   E5 (different topic) has lower similarity with all")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Multilingual Semantic Similarity:

Text                                               E1       F2       S3       J4       E5
Machine learning is transforming AI                1.00 0.95 0.95 0.94 -0.09
L'apprentissage automatique transforme l'IA        0.95 1.00 0.96 0.93 -0.07
El aprendizaje automático está transformando la    0.95 0.96 1.00 0.94 -0.08
機械学習はAIを変革しています                                    0.94 0.93 0.94 1.00 -0.10
The weather is nice today                          -0.09 -0.07 -0.08 -0.10 1.00

💡 E1, F2, S3, J4 have high similarity - same meaning, different languages!
   E5 (different topic) has lower similarity with all


## 8. Domain-Specific Embeddings

In [10]:
# Different models for different domains
general_model = SentenceTransformer('all-MiniLM-L6-v2')
qa_model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

# Question-answer pair
question = "What is the capital of France?"
answer = "The capital of France is Paris."
random_text = "Machine learning is a subset of artificial intelligence."

texts = [question, answer, random_text]

# Compare models
print("General Model vs QA-Optimized Model:\n")

for model_name, model in [("General", general_model), ("QA-Optimized", qa_model)]:
    embs = model.encode(texts)
    q_a_sim = cosine_similarity([embs[0]], [embs[1]])[0][0]
    q_r_sim = cosine_similarity([embs[0]], [embs[2]])[0][0]
    
    print(f"{model_name}:")
    print(f"  Question-Answer similarity: {q_a_sim:.4f}")
    print(f"  Question-Random similarity: {q_r_sim:.4f}")
    print(f"  Difference: {q_a_sim - q_r_sim:.4f}\n")

print("💡 QA-optimized models better distinguish relevant from irrelevant!")

General Model vs QA-Optimized Model:

General:
  Question-Answer similarity: 0.8790
  Question-Random similarity: 0.0506
  Difference: 0.8284

QA-Optimized:
  Question-Answer similarity: 0.8853
  Question-Random similarity: 0.0719
  Difference: 0.8134

💡 QA-optimized models better distinguish relevant from irrelevant!


## 9. Production-Ready Dense Retriever

In [11]:
class ProductionDenseRetriever:
    def __init__(self, model_name='all-MiniLM-L6-v2', normalize=True):
        self.model = SentenceTransformer(model_name)
        self.normalize = normalize
        self.documents = []
        self.embeddings = None
        self.metadata = []
        
    def index(self, documents, metadata=None, batch_size=32):
        """Index documents with optional metadata"""
        self.documents = documents
        self.metadata = metadata if metadata else [{} for _ in documents]
        
        # Encode with batching
        self.embeddings = self.model.encode(
            documents,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=self.normalize
        )
        
        print(f"✅ Indexed {len(documents)} documents")
        
    def search(self, query, top_k=5, score_threshold=None):
        """Search with optional score filtering"""
        # Encode query
        query_embedding = self.model.encode(
            query, 
            normalize_embeddings=self.normalize
        )
        
        # Calculate scores (dot product if normalized, else cosine)
        if self.normalize:
            scores = np.dot(self.embeddings, query_embedding)
        else:
            scores = cosine_similarity([query_embedding], self.embeddings)[0]
        
        # Filter by threshold
        if score_threshold:
            mask = scores >= score_threshold
            valid_indices = np.where(mask)[0]
            scores = scores[mask]
        else:
            valid_indices = np.arange(len(scores))
        
        # Get top-k
        top_k = min(top_k, len(scores))
        top_positions = np.argsort(scores)[::-1][:top_k]
        top_indices = valid_indices[top_positions]
        
        # Build results
        results = []
        for rank, idx in enumerate(top_indices):
            results.append({
                'rank': rank + 1,
                'score': float(scores[top_positions[rank]]),
                'document': self.documents[idx],
                'metadata': self.metadata[idx],
                'index': int(idx)
            })
        
        return results
    
    def save(self, path):
        """Save index to disk"""
        np.savez(
            path,
            embeddings=self.embeddings,
            documents=self.documents,
            metadata=self.metadata
        )
        print(f"✅ Index saved to {path}")
    
    def load(self, path):
        """Load index from disk"""
        data = np.load(path, allow_pickle=True)
        self.embeddings = data['embeddings']
        self.documents = data['documents'].tolist()
        self.metadata = data['metadata'].tolist()
        print(f"✅ Index loaded from {path}")

# Test production retriever
prod_retriever = ProductionDenseRetriever()

# Documents with metadata
docs_with_meta = [
    ("Python is great for data science", {"source": "blog", "date": "2024-01"}),
    ("Machine learning transforms industries", {"source": "paper", "date": "2024-02"}),
    ("RAG improves LLM accuracy", {"source": "blog", "date": "2024-03"}),
]

docs = [d[0] for d in docs_with_meta]
metadata = [d[1] for d in docs_with_meta]

prod_retriever.index(docs, metadata=metadata)

# Search
results = prod_retriever.search("AI and machine learning", top_k=2)

print("\nSearch Results:")
for r in results:
    print(f"\nRank {r['rank']}: {r['document']}")
    print(f"  Score: {r['score']:.4f}")
    print(f"  Metadata: {r['metadata']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Indexed 3 documents

Search Results:

Rank 1: Machine learning transforms industries
  Score: 0.5448
  Metadata: {'source': 'paper', 'date': '2024-02'}

Rank 2: Python is great for data science
  Score: 0.1761
  Metadata: {'source': 'blog', 'date': '2024-01'}


## Key Takeaways 🎯

### ✅ Dense Retrieval Essentials:

1. **Embeddings** convert text to dense vectors
2. **Semantic similarity** finds meaning, not just keywords
3. **Cosine similarity** is standard metric
4. **Batch processing** is crucial for speed
5. **Model choice** affects quality and speed

### 🎯 Model Selection Guide:

```python
# Fast and efficient (384 dims)
'all-MiniLM-L6-v2'  # ✅ Best for most RAG systems

# Better quality (768 dims)
'all-mpnet-base-v2'  # Higher accuracy, slower

# QA optimized
'multi-qa-MiniLM-L6-cos-v1'  # Best for Q&A

# Multilingual
'paraphrase-multilingual-MiniLM-L12-v2'  # 50+ languages

# State-of-the-art
'intfloat/e5-base-v2'  # Current best open-source
```

### 📊 Performance Tips:

1. **Always use batching**:
   ```python
   embeddings = model.encode(docs, batch_size=32)
   ```

2. **Normalize embeddings**:
   ```python
   embeddings = model.encode(docs, normalize_embeddings=True)
   # Then use dot product instead of cosine (faster!)
   ```

3. **Cache embeddings**:
   ```python
   np.save('embeddings.npy', embeddings)  # Save
   embeddings = np.load('embeddings.npy')  # Load
   ```

4. **GPU acceleration**:
   ```python
   model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
   ```

### 🚀 Production Checklist:

- [ ] Choose appropriate model for domain
- [ ] Use batch processing for indexing
- [ ] Normalize embeddings if using dot product
- [ ] Cache/save embeddings to disk
- [ ] Add metadata support
- [ ] Implement score thresholding
- [ ] Use GPU if available
- [ ] Monitor inference speed

### 💡 Common Patterns:

```python
# Pattern 1: Basic RAG
retriever = DenseRetriever('all-MiniLM-L6-v2')
retriever.index(documents)
results = retriever.search(query, top_k=5)

# Pattern 2: With filtering
results = retriever.search(query, score_threshold=0.5)

# Pattern 3: Multilingual
retriever = DenseRetriever('paraphrase-multilingual-MiniLM-L12-v2')
```

## Next Steps 🔜

Move to `02_Sparse_Retrieval.ipynb` to learn BM25 and TF-IDF!

Then we'll combine both in `03_Hybrid_Retrieval.ipynb` for best results! 🚀